# RQ2 RP-Geo — freeze one retention (CPU only)

This is a separate scientific freeze stage. It reads the completed Stage A/B Pareto probe, shows the mechanistic effect sizes, and writes one immutable retention artifact. It never trains a model and never reads accuracy.

## Required input

Attach the output of `kaggle_rq2_rpgeo_extension_t4x2.ipynb` (normally `rq2-rpgeo-stage-ab-v1.zip`) and define Kaggle secret `github_token`. No CIFAR-100 and no GPU are needed.

In [ ]:
import os, subprocess, sys, json, zipfile, importlib, hashlib
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

In [ ]:
import pandas as pd
import rq2_e2e_pairwise_pilot as pilot
import rq2_rpgeo_gate as gate
pilot = importlib.reload(pilot); gate = importlib.reload(gate)
INPUT_ROOT = Path('/kaggle/input')
PROBE_NAMES = ('rpgeo_retention_probe.json','rpgeo_retention_pareto.csv','rpgeo_retention_probe_by_state.csv')
def probe_signature(folder): return tuple(hashlib.sha256((folder/name).read_bytes()).hexdigest() for name in PROBE_NAMES)
direct = sorted({path.parent for path in INPUT_ROOT.rglob(PROBE_NAMES[0]) if all((path.parent/name).is_file() for name in PROBE_NAMES)})
if direct:
    groups = {}
    for folder in direct: groups.setdefault(probe_signature(folder), []).append(folder)
    assert len(groups)==1, f'Multiple content-distinct probes: {direct}'
    PROBE_DIR = sorted(next(iter(groups.values())),key=lambda p:(len(str(p)),str(p)))[0]
else:
    archived = {}
    for archive in INPUT_ROOT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive) as z:
                members = {name:[item for item in z.namelist() if item.endswith('rpgeo_retention_probe/'+name)] for name in PROBE_NAMES}
                if not all(len(items)==1 for items in members.values()): continue
                blobs = {name:z.read(items[0]) for name,items in members.items()}
                signature = tuple(hashlib.sha256(blobs[name]).hexdigest() for name in PROBE_NAMES)
                archived[signature] = (archive,blobs)
        except (OSError,zipfile.BadZipFile): pass
    assert len(archived)==1, f'Expected one content-unique Stage-A/B probe; found {len(archived)}'
    _,blobs = next(iter(archived.values())); PROBE_DIR = Path('/kaggle/working/materialized-rpgeo-probe'); PROBE_DIR.mkdir(parents=True,exist_ok=True)
    for name,blob in blobs.items(): (PROBE_DIR/name).write_bytes(blob)
required = [PROBE_DIR/'rpgeo_retention_probe.json', PROBE_DIR/'rpgeo_retention_pareto.csv', PROBE_DIR/'rpgeo_retention_probe_by_state.csv']
assert all(path.is_file() for path in required), f'Missing Stage A/B probe: {[str(p) for p in required if not p.is_file()]}'
probe = json.loads(required[0].read_text())
assert probe['selection_uses_accuracy'] is False and probe['training_authorized'] is False
pareto = pd.read_csv(required[1]).sort_values('resource_retention_target', ascending=False)
if 'mean_resource_sacrifice' not in pareto.columns:
    by_state = pd.read_csv(required[2]); by_state['resource_sacrifice'] = 1.0-by_state.resource_retention_achieved
    sacrifice = by_state.groupby('resource_retention_target').resource_sacrifice.agg(mean_resource_sacrifice='mean',maximum_resource_sacrifice='max').reset_index()
    pareto = pareto.merge(sacrifice,on='resource_retention_target',how='left')
display_columns = ['resource_retention_target','minimum_resource_retention_achieved','mean_resource_sacrifice','maximum_resource_sacrifice','variance_win_states','movement_states','mean_variance_delta','worst_variance_delta','mean_oracle_gap_captured','minimum_oracle_gap_captured','all_state_mechanistic_pass']
missing_optional = [name for name in ('mean_oracle_gap_captured','minimum_oracle_gap_captured') if name not in pareto.columns]
if missing_optional: print('Older Stage-A/B artifact; optional metrics unavailable:',missing_optional)
display(pareto[[name for name in display_columns if name in pareto.columns]])
print('Pre-registered mathematical candidate:', probe.get('mechanistic_candidate_retention'))

## Human freeze decision

`SELECTED_RETENTION` is pre-registered as `0.995` from the mechanistic Pareto evidence. This freeze remains a separate stage from E2E training, and the value must never be changed after viewing Stage-C accuracy.

In [ ]:
SELECTED_RETENTION = 0.995  # frozen from mechanistic evidence; never change after Stage-C accuracy
FREEZE_PATH = Path('/kaggle/working/rpgeo_frozen_retention.json')
matches = pareto.loc[__import__('numpy').isclose(pareto.resource_retention_target,SELECTED_RETENTION)]
assert len(matches)==1, 'rho=0.995 is absent from the pre-enumerated Pareto grid'
selected = matches.iloc[0]
passed = selected.all_state_mechanistic_pass
if isinstance(passed,str): passed = passed.strip().lower()=='true'
assert bool(passed), 'rho=0.995 did not pass the frozen 5/5 mechanistic gate'
by_state = pd.read_csv(required[2]); state_rows = by_state.loc[__import__('numpy').isclose(by_state.resource_retention_target,SELECTED_RETENTION)].copy()
wins = state_rows.variance_win.map(lambda value: value if isinstance(value,bool) else str(value).strip().lower()=='true')
assert len(state_rows)==5 and wins.all()
sacrifice = 1.0-state_rows.resource_retention_achieved.to_numpy(float)
source_hashes = {'probe_json_sha256':required[0],'probe_pareto_sha256':required[1],'probe_by_state_sha256':required[2]}
frozen = {
 'status':'RPGEO_RETENTION_FROZEN_BEFORE_E2E', 'resource_retention':0.995,
 'selection_source':'mechanistic_pareto_development', 'selection_frozen':True,
 'frozen_before_rpgeo_e2e':True, 'accuracy_used':False,
 'selection_rule':'pre-registered rho=0.995 from the development mechanistic Pareto probe',
 'freeze_git_commit':GIT_COMMIT, 'mechanistic_candidate_retention':probe.get('mechanistic_candidate_retention'),
 'selected_probe_metrics':{
   'variance_win_states':int(selected.variance_win_states),'movement_states':int(selected.movement_states),
   'mean_variance_delta':float(selected.mean_variance_delta),'worst_variance_delta':float(selected.worst_variance_delta),
   'oracle_gap_captured_available':bool('mean_oracle_gap_captured' in pareto.columns),
   'mean_oracle_gap_captured':float(selected.mean_oracle_gap_captured) if 'mean_oracle_gap_captured' in pareto.columns else None,
   'minimum_oracle_gap_captured':float(selected.minimum_oracle_gap_captured) if 'minimum_oracle_gap_captured' in pareto.columns else None,
   'minimum_resource_retention_achieved':float(selected.minimum_resource_retention_achieved),
   'mean_resource_sacrifice':float(sacrifice.mean()),'maximum_resource_sacrifice':float(sacrifice.max())},
 **{key:hashlib.sha256(path.read_bytes()).hexdigest() for key,path in source_hashes.items()}
}
if FREEZE_PATH.exists(): assert json.loads(FREEZE_PATH.read_text())==frozen, 'A different retention is already frozen'
else: FREEZE_PATH.write_text(json.dumps(frozen,indent=2)+'\n')
assert frozen['resource_retention'] == 0.995
assert frozen['selection_source'] == 'mechanistic_pareto_development'
assert frozen['accuracy_used'] is False and frozen['frozen_before_rpgeo_e2e'] is True
print(json.dumps(frozen, indent=2))

In [ ]:
# Revalidate against the untouched probe before export.
# The artifact is hash-bound to all three probe files; Stage C revalidates it against its Stage-A/B root.
for key,path in source_hashes.items(): assert frozen[key]==hashlib.sha256(path.read_bytes()).hexdigest()
bundle = Path('/kaggle/working/rq2-rpgeo-retention-freeze-v1.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(FREEZE_PATH, 'rpgeo_frozen_retention.json')
print('Persist both outputs:', FREEZE_PATH, bundle)
bundle